## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.m

In [1]:
import os
from dotenv import load_dotenv

groq_api_key=os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq
model=ChatGroq(model='llama-3.1-8b-instant', api_key=groq_api_key, temperature=0.1)
model

c:\Users\Dell\Desktop\codes\Udemy_GenAI\LangChain\lang\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000209F8BA8FD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000209F8BA96C0>, model_name='llama-3.1-8b-instant', temperature=0.1, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content='Hi I am Dhyan Sher and I am a chief AI Engineer')])

AIMessage(content="Nice to meet you, Dhyan Sher. As a chief AI Engineer, I'm sure you have a deep understanding of the latest advancements in artificial intelligence and its applications. What specific areas of AI are you currently working on or interested in?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 48, 'total_tokens': 97, 'completion_time': 0.063576853, 'completion_tokens_details': None, 'prompt_time': 0.004342301, 'prompt_tokens_details': None, 'queue_time': 0.049167114, 'total_time': 0.067919154}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a00408-abc7-76c0-8028-b537929601b2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 49, 'total_tokens': 97})

In [4]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Dhyan and I am a Chief AI Engineer"),
        AIMessage(content="Hello Dhyan! It's nice to meet you. \n\nAs a Chief AI Engineer, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content="Your name is Dhyan, and you're a Chief AI Engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 119, 'total_tokens': 134, 'completion_time': 0.017216496, 'completion_tokens_details': None, 'prompt_time': 0.019153304, 'prompt_tokens_details': None, 'queue_time': 0.046906879, 'total_time': 0.0363698}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a00408-af87-7e73-8e3f-9a6b0ea0853a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 119, 'output_tokens': 15, 'total_tokens': 134})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory(session_id=session_id)
    return store[session_id]

with_history = RunnableWithMessageHistory(model, get_session_history=get_session_history)

C:\Users\Dell\AppData\Local\Temp\ipykernel_19556\2004652707.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
c:\Users\Dell\Desktop\codes\Udemy_GenAI\LangChain\lang\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
config={'configurable':{'session_id':'chat1'}}

In [7]:
response=with_history.invoke([
    HumanMessage(content="Hi , My name is Dhyan and I am a Chief AI Engineer"),
    AIMessage(content="Hello Dhyan! It's nice to meet you. \n\nAs a Chief AI Engineer, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
    HumanMessage(content="Hey What's my name and what do I do?")
], config=config)

In [8]:
response.content

"Your name is Dhyan, and you're a Chief AI Engineer."

In [9]:
with_history.invoke([
    HumanMessage(content='Whats my name?')],config=config
)

AIMessage(content='Your name is Dhyan.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 147, 'total_tokens': 154, 'completion_time': 0.012909723, 'completion_tokens_details': None, 'prompt_time': 0.00834154, 'prompt_tokens_details': None, 'queue_time': 0.158023994, 'total_time': 0.021251263}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a00408-b32b-7e40-81eb-5bec2e4d8e15-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 147, 'output_tokens': 7, 'total_tokens': 154})

In [10]:
## change the session id to chat2 and check the history is not there
config={'configurable':{'session_id':'chat2'}}

In [11]:
response=with_history.invoke([
    HumanMessage(content="Hi whats my name?")], config=config)

In [12]:
response.content

"I don't have any information about your name. I'm a large language model, I don't have the ability to retain information about individual users or their personal details. Each time you interact with me, it's a new conversation and I don't have any prior knowledge about you. If you'd like to share your name with me, I'd be happy to chat with you!"

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [13]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages([
    ('system','You are a helpful assistant.Answer the question based on the context below. If the question is not related to the context, politely respond that you are tuned to only answer questions that are related to the context in {language}.'),
    MessagesPlaceholder(variable_name="messages"),
])
chain=prompt|model

In [14]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Dhyan")], "language":"Hindi"})
response.content

'नमस्ते ध्यान जी, मैं आपकी सहायता करने के लिए यहाँ हूँ। क्या मैं आपकी किसी समस्या या प्रश्न का समाधान करने में मदद कर सकता हूँ?'

In [15]:
with_history=RunnableWithMessageHistory(chain, get_session_history=get_session_history,input_messages_key="messages", output_key="response")

c:\Users\Dell\Desktop\codes\Udemy_GenAI\LangChain\lang\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [16]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Dhyan")],"language":"Hindi"},
    config=config
)
repsonse.content

'नमस्ते ध्यान जी, मैं आपकी सहायता के लिए यहाँ हूँ। क्या मैं आपकी किसी समस्या या प्रश्न का समाधान करने में मदद कर सकता हूँ?'

In [17]:
response = with_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

In [18]:
response.content

'आपका नाम ध्यान है।'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [19]:
from langchain_core.messages import SystemMessage,trim_messages

In [21]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

c:\Users\Dell\Desktop\codes\Udemy_GenAI\LangChain\lang\lib\site-packages\langchain_core\language_models\base.py:448: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))
c:\Users\Dell\Desktop\codes\Udemy_GenAI\LangChain\lang\lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate D

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [22]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
chain=(
RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer)|prompt|model

)

response=chain.invoke({"messages":messages+[HumanMessage(content="what icecream do i like?")], "language":"Hindi"})
response.content

'Main aapki ice cream ki preference ke baare mein jaankari nahin rakhta hoon.'

In [23]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked the math problem "whats 2 + 2".'

In [24]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

c:\Users\Dell\Desktop\codes\Udemy_GenAI\LangChain\lang\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [25]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

"I don't have any information about your name. I'm tuned to only answer questions that are related to the context, which is our conversation about being a good assistant."

In [26]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

"You didn't ask a math problem in the given context. The context only mentions that you're a good assistant, but there's no mention of a math problem. I'm tuned to only answer questions that are related to the context in English."